# 03 · Benchmark: pose models vs. mocap
<a href="https://colab.research.google.com/github/danielamrh/mmc-upper-limb/blob/main/notebooks/03_benchmark.ipynb"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open in Colab"/></a>

Compares the cached pose output (notebook 02 / `tools/run_pose.py`) with the OptiTrack ground truth,
repetition by repetition: joint angles (no alignment needed), MPJPE and the movement quality measures
after one similarity alignment per video (oracle scale), split by viewpoint.
**Runtime: CPU is enough.**

## 0 · Drive + repo + install
Code comes from GitHub (always fresh via `git pull`), data and results live on Google Drive.

In [ ]:
import os, sys
from google.colab import drive
drive.mount("/content/drive")

REPO = "mmc-upper-limb"
REPO_DIR = f"/content/{REPO}"
if not os.path.exists(REPO_DIR):
    !git clone -q https://github.com/danielamrh/{REPO}.git {REPO_DIR}
else:
    !git -C {REPO_DIR} pull -q
%cd {REPO_DIR}
COMMIT = !git rev-parse --short HEAD
COMMIT = COMMIT[0]
print("commit:", COMMIT)

!pip install -q -e .
sys.path.insert(0, f"{REPO_DIR}/src")  # editable install is only picked up after a restart
# re-running this cell must pick up freshly pulled code
for name in [m for m in sys.modules if m == "mmcul" or m.startswith("mmcul.")]:
    del sys.modules[name]

DRIVE_ROOT = "/content/drive/MyDrive/mmc_upper_limb"
os.makedirs(DRIVE_ROOT, exist_ok=True)

In [ ]:
!pip install -q -e ".[eval]"
from pathlib import Path
import pandas as pd, numpy as np, matplotlib.pyplot as plt
from mmcul.benchmark import evaluate_model, agreement_table, TUNING_PERSONS
from mmcul.datasets import rehab24

DATA = Path("/content/data/rehab24")
CACHE = Path(DRIVE_ROOT) / "cache" / "rehab24"
# the ground truth is small; videos are not needed here
rehab24.download(["Segmentation.csv"], DATA)
for z in ("3d_joints.zip",):
    rehab24.fetch_members(z, DATA, (1, 2), lambda n: n.endswith("-30fps.npy"))
MODELS = sorted(p.name for p in CACHE.iterdir() if p.is_dir())
print("cached models:", MODELS)

## 1 · Evaluate every cached model

In [ ]:
frames = []
for m in MODELS:
    # persons used to tune the biomechanical fit are excluded for every model
    rows = evaluate_model(DATA, CACHE, m, exclude_persons=TUNING_PERSONS)
    print(f"{m}: {len(rows)} repetition x camera rows, {len(rows.missing)} videos not cached yet")
    frames.append(pd.DataFrame(rows))
df = pd.concat(frames, ignore_index=True)
out = Path(DRIVE_ROOT) / "results" / f"benchmark_rehab24_{COMMIT}.csv"
out.parent.mkdir(parents=True, exist_ok=True)
df.to_csv(out, index=False)
print("saved", out)

## 2 · Pose accuracy by viewpoint

In [ ]:
cols = ["mpjpe_mm", "rmse_elbow_flexion", "rmse_shoulder_elevation", "r_elbow_flexion", "r_shoulder_elevation", "valid_frac"]
display(df.groupby(["model", "orientation"])[cols].median().round(2))

## 3 · Agreement of the movement quality measures
Bias and 95 % limits of agreement (Bland-Altman), absolute-agreement ICC(A,1) and Pearson r, per model and viewpoint.

In [ ]:
tab = agreement_table(df)
pd.set_option("display.width", 200)
display(tab.pivot_table(index=["measure"], columns=["model", "orientation"], values="icc_a1").round(2))
display(tab[["model", "orientation", "measure", "n", "bias", "loa_low", "loa_high", "mae", "r", "icc_a1"]].round(3))

## 4 · Bland-Altman plots

In [ ]:
measures = ["peak_speed", "n_movement_units", "trunk_displacement", "min_elbow_flexion", "max_shoulder_elevation"]
for m_name in MODELS:
    d = df[df.model == m_name]
    fig, axes = plt.subplots(1, len(measures), figsize=(4 * len(measures), 3.5))
    for ax, m in zip(axes, measures):
        for o, g in d.groupby("orientation"):
            mean = (g[f"pred_{m}"] + g[f"gt_{m}"]) / 2
            ax.scatter(mean, g[f"pred_{m}"] - g[f"gt_{m}"], s=8, alpha=.6, label=o)
        ax.axhline(0, c="k", lw=.8); ax.set_title(m, fontsize=9); ax.set_xlabel("mean"); ax.set_ylabel("pred - gt")
    axes[0].legend(fontsize=8); fig.suptitle(m_name); plt.tight_layout()